In [1]:
# ============================================================
# UPDATED: generator takes a discharge DataFrame (datetime index)
# and returns synthetics as DataFrames with the SAME datetime index.
# ============================================================

import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


def generate_synthetic_hydrographs_from_real(
    Q_df,                  # pandas DataFrame or Series with datetime index
    value_col="discharge_m3s",
    n=500,
    seed=0,
    enforce_peak_range=True,

    # peak range rule (relative to real max)
    peak_factor_range=(0.70, 1.20),   # A=0.70*Qmax, B=1.20*Qmax

    # template smoothing
    smooth_real=True,
    real_smooth_window=31,
    real_smooth_poly=3,

    # time warp
    warp_a_range=(0.75, 1.35),
    t_split_range=(0.20, 0.40),

    # main shape edits
    peak_scale_range=(0.90, 1.15),
    peak_sharp_range=(0.90, 1.25),
    recession_k_range=(0.85, 1.20),

    # tail control
    floor_factor_range=(0.90, 1.80),
    floor_strength=4.0,
    end_offset_range=(-0.02, 0.10),
    end_offset_strength=4.0,

    # final smoothing
    smooth_syn_window=23,
    smooth_syn_poly=3,

    # double peak controls
    p_double=0.7,
    dt2_range=(0.035, 0.10),
    dip_dt_range=(0.015, 0.045),
    dip_depth_range=(0.10, 0.25),
    peak2_amp_range=(0.15, 0.35),
    width_dip_range=(0.010, 0.030),
    width_peak2_range=(0.015, 0.045),
):
    """
    Input
    -----
    Q_df : pandas.DataFrame or pandas.Series
        Must have datetime index. If DataFrame, uses `value_col`.

    Returns
    -------
    Q_syn_df : pandas.DataFrame
        Synthetic discharge with SAME datetime index.
        Columns: syn_000, syn_001, ..., syn_{n-1}
    Q_norm_df : pandas.DataFrame
        Same synthetics normalized to [0,1] using (Qmin,Qmax) from the *smoothed real*.
        Same datetime index and column names.
    meta : dict
        Contains: Qmin, Qmax, base, peak_range_m3s, seed, n
    """

    rng = np.random.default_rng(seed)

    # ---- coerce input to Series with datetime index
    if isinstance(Q_df, pd.Series):
        q_series = Q_df.copy()
    elif isinstance(Q_df, pd.DataFrame):
        if value_col not in Q_df.columns:
            raise ValueError(f"value_col='{value_col}' not found in Q_df.columns={list(Q_df.columns)}")
        q_series = Q_df[value_col].copy()
    else:
        raise TypeError("Q_df must be a pandas DataFrame or Series with a datetime index.")

    if not isinstance(q_series.index, pd.DatetimeIndex):
        raise TypeError("Q_df must have a DatetimeIndex.")

    # ---- numeric array + fill missing
    Q = q_series.to_numpy(dtype=float)
    if not np.all(np.isfinite(Q)):
        idx = np.arange(len(Q))
        good = np.isfinite(Q)
        Q = np.interp(idx, idx[good], Q[good])

    # ---- smooth the real series for template
    Q_sm = Q.copy()
    if smooth_real and len(Q_sm) >= 7:
        w = min(real_smooth_window, len(Q_sm) - (1 - len(Q_sm) % 2))
        if w < 7:
            w = 7 if len(Q_sm) >= 7 else (len(Q_sm) // 2) * 2 + 1
        if w % 2 == 0:
            w += 1
        Q_sm = savgol_filter(Q_sm, w, min(real_smooth_poly, w - 1))

    nt = len(Q_sm)
    t = np.linspace(0.0, 1.0, nt)

    Qmin = float(np.min(Q_sm))
    Qmax = float(np.max(Q_sm))

    q0 = (Q_sm - Qmin) / (Qmax - Qmin + 1e-12)
    q_start = float(q0[0])

    # IMPORTANT: baseflow anchor should match ORIGINAL series start, not smoothed
    base = float(Q[0])

    # peak range computed from REAL max (original, not smoothed)
    real_peak = float(np.max(Q))
    A = peak_factor_range[0] * real_peak
    B = peak_factor_range[1] * real_peak
    peak_range_m3s = (A, B)

    # ---- internal helpers
    def time_warp(t, a, t_split):
        s_split = np.clip(a * t_split, 1e-6, 1 - 1e-6)
        slope2 = (1.0 - s_split) / (1.0 - t_split)
        s = np.empty_like(t)
        m1 = t <= t_split
        s[m1] = a * t[m1]
        s[~m1] = s_split + slope2 * (t[~m1] - t_split)
        return np.clip(s, 0.0, 1.0)

    def gauss(t, c, w, a):
        return a * np.exp(-0.5 * ((t - c) / (w + 1e-12))**2)

    def smooth_syn(y):
        if nt < 7:
            return y
        w = min(smooth_syn_window, nt - (1 - nt % 2))
        if w < 7:
            w = 7 if nt >= 7 else (nt // 2) * 2 + 1
        if w % 2 == 0:
            w += 1
        return savgol_filter(y, w, min(smooth_syn_poly, w - 1))

    # -------------------------------------------------
    # 1) generate normalized synthetics
    # -------------------------------------------------
    syn_norm = np.zeros((n, nt), dtype=float)

    for i in range(n):
        a = rng.uniform(*warp_a_range)
        t_split = rng.uniform(*t_split_range)
        s = time_warp(t, a, t_split)
        q = np.interp(s, t, q0)

        q = np.clip(rng.uniform(*peak_scale_range) * q, 0.0, 2.0)
        q = np.clip(q, 0.0, 1.0) ** (1.0 / rng.uniform(*peak_sharp_range))

        j1 = int(np.argmax(q))
        t1 = t[j1]

        if rng.random() < p_double:
            t_dip = np.clip(t1 + rng.uniform(*dip_dt_range), 0.05, 0.95)
            t2 = np.clip(t1 + rng.uniform(*dt2_range), 0.05, 0.97)
            if t2 <= t_dip + 0.01:
                t2 = np.clip(t_dip + 0.02, 0.05, 0.97)

            q -= gauss(t, t_dip, rng.uniform(*width_dip_range), rng.uniform(*dip_depth_range))
            q += gauss(t, t2,    rng.uniform(*width_peak2_range), rng.uniform(*peak2_amp_range))

        q = np.clip(q, 0.0, 1.0)

        j_peak = int(np.argmax(q))
        k_rec = rng.uniform(*recession_k_range)
        if j_peak < nt - 2:
            tt = np.linspace(0.0, 1.0, nt - j_peak)
            q[j_peak:] *= (1.0 - tt) ** k_rec

        q[0] = q_start

        q_floor = rng.uniform(*floor_factor_range) * q_start
        w_floor = (1.0 - q) ** floor_strength
        w_floor[:j_peak] = 0.0
        q += w_floor * np.maximum(0.0, q_floor - q)

        w_end = (1.0 - q) ** end_offset_strength
        w_end[:j_peak] = 0.0
        q += w_end * rng.uniform(*end_offset_range)

        q = smooth_syn(np.clip(q, 0.0, 1.0))
        q[0] = q_start

        syn_norm[i] = np.clip(q, 0.0, 1.0)

    # -------------------------------------------------
    # 2) normalized -> discharge
    # -------------------------------------------------
    syn_Q = Qmin + syn_norm * (Qmax - Qmin)
    syn_Q[:, 0] = base

    # -------------------------------------------------
    # 3) enforce peak range in discharge units (optional)
    # -------------------------------------------------
    if enforce_peak_range:
        if B <= A:
            raise ValueError(f"Invalid peak range computed: {peak_range_m3s}")
        if B <= base:
            raise ValueError(f"Peak upper bound B={B:.3f} must be > baseflow Q[0]={base:.3f}")

        peaks_now = syn_Q.max(axis=1)
        target_peaks = rng.uniform(A, B, size=n)

        scale = (target_peaks - base) / (peaks_now - base + 1e-12)
        scale = np.clip(scale, 0.1, 10.0)

        syn_Q = base + (syn_Q - base) * scale[:, None]
        syn_Q[:, 0] = base

        # keep normalized consistent with final syn_Q
        syn_norm = (syn_Q - Qmin) / (Qmax - Qmin + 1e-12)
        syn_norm = np.clip(syn_norm, 0.0, 1.0)

    # -------------------------------------------------
    # 4) pack as DataFrames with SAME datetime index
    # -------------------------------------------------
    idx = q_series.index
    cols = [f"syn_{i:03d}" for i in range(n)]

    Q_syn_df = pd.DataFrame(syn_Q.T, index=idx, columns=cols)
    Q_norm_df = pd.DataFrame(syn_norm.T, index=idx, columns=cols)

    meta = dict(
        Qmin=Qmin,
        Qmax=Qmax,
        base=base,
        peak_range_m3s=peak_range_m3s,
        seed=seed,
        n=n,
        value_col=value_col,
    )

    return Q_syn_df, Q_norm_df, meta


In [2]:
import os
import geopandas as gpd
import pandas as pd
import numpy as np

def load_gauge_and_discharge(
    gauge_shp_dir,
    gauge_shp_name,
    discharge_folder,
    gauge_index=None,                    # None = all gauges, int = single gauge
    target_crs="EPSG:32618",
    default_crs="EPSG:4326",
):
    """
    Load gauge locations and corresponding discharge time series.

    Parameters
    ----------
    gauge_shp_dir : str
        Directory containing the gauge shapefile.
    gauge_shp_name : str
        Name of the gauge shapefile (e.g., "UpstreamBoundry_locations.shp").
    discharge_folder : str
        Folder containing discharge CSVs named as {usgs_id}.csv.
    gauge_index : int or None, optional
        Index of gauge to load (0-based). If None, load ALL gauges.
    target_crs : str, default "EPSG:32618"
        Desired CRS for geometry (if needed).
    default_crs : str, default "EPSG:4326"
        Fallback CRS if shapefile has none.

    Returns
    -------
    gauges : GeoDataFrame
        Full gauge table (with geometry, projected if requested).
    labels : list[str] or str
        Gauge label(s). List if all gauges, str if single.
    usgs_ids : list[str] or str
        Parsed USGS ID(s). List if all, str if single.
    discharge_dfs : list[pd.DataFrame] or pd.DataFrame
        Discharge DataFrame(s) with datetime index. List if all, single if one.
    Q_arrays : list[np.ndarray] or np.ndarray
        Discharge values as numpy array(s). List if all, array if single.
    nt_values : list[int] or int
        Number of time steps per hydrograph.
    t_days_arrays : list[np.ndarray] or np.ndarray
        Time vectors in days from start of each record.
    """
    
    # -----------------------------
    # Load and prepare gauge shapefile
    # -----------------------------
    shp_path = os.path.join(gauge_shp_dir, gauge_shp_name)
    if not os.path.exists(shp_path):
        raise FileNotFoundError(f"Gauge shapefile not found: {shp_path}")

    gauges = gpd.read_file(shp_path)

    if gauges.crs is None:
        print("Warning: No CRS found in shapefile. Assigning default:", default_crs)
        gauges = gauges.set_crs(default_crs)

    if str(gauges.crs) != target_crs:
        try:
            gauges = gauges.to_crs(target_crs)
            print(f"Projected gauges to {target_crs}")
        except Exception as e:
            print(f"Warning: Failed to project to {target_crs}: {e}")

    # Determine which rows to process
    if gauge_index is not None:
        if gauge_index < 0 or gauge_index >= len(gauges):
            raise IndexError(f"gauge_index={gauge_index} out of range (0..{len(gauges)-1})")
        rows = [gauges.iloc[[gauge_index]]]  # Keep as list of 1-item GeoDataFrames
        single_mode = True
    else:
        rows = [gauges.iloc[[i]] for i in range(len(gauges))]
        single_mode = False

    # -----------------------------
    # Process each selected gauge
    # -----------------------------
    labels = []
    usgs_ids = []
    discharge_dfs = []
    Q_arrays = []
    nt_values = []
    t_days_arrays = []

    for idx, row_gdf in enumerate(rows):
        row = row_gdf.iloc[0]

        # Extract label
        if "Name" in gauges.columns and pd.notna(row.get("Name")):
            label = str(row["Name"])
        elif "ID" in gauges.columns and pd.notna(row.get("ID")):
            label = str(row["ID"])
        else:
            label = f"gauge_{idx}"

        # Extract USGS ID (last part after underscore)
        usgs_id = label.split("_")[-1]

        # Load discharge CSV
        csv_path = os.path.join(discharge_folder, f"{usgs_id}.csv")
        if not os.path.exists(csv_path):
            raise FileNotFoundError(f"Discharge file not found: {csv_path}")

        df = pd.read_csv(csv_path, header=0, names=["datetime", "discharge_m3s"])
        df["datetime"] = pd.to_datetime(df["datetime"])
        df = df.set_index("datetime").sort_index()

        # Remove any rows with NaN discharge (optional, but safe)
        df = df.dropna(subset=["discharge_m3s"])

        Q_arr = df["discharge_m3s"].to_numpy(dtype=float)
        nt = len(Q_arr)

        # Time in days from start of this record
        start_time = df.index[0]
        t_seconds = (df.index - start_time).total_seconds()
        t_days_arr = t_seconds / (24 * 3600)

        # Append results
        labels.append(label)
        usgs_ids.append(usgs_id)
        discharge_dfs.append(df)
        Q_arrays.append(Q_arr)
        nt_values.append(nt)
        t_days_arrays.append(t_days_arr)

    # -----------------------------
    # Return in consistent format (single or list)
    # -----------------------------
    if single_mode:
        return (
            gauges,
            labels[0],
            usgs_ids[0],
            discharge_dfs[0],
            Q_arrays[0],
            nt_values[0],
            t_days_arrays[0],
        )
    else:
        return (
            gauges,
            labels,
            usgs_ids,
            discharge_dfs,
            Q_arrays,
            nt_values,
            t_days_arrays,
        )

In [3]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


def generate_synthetic_hydrographs_from_real(
    Q_df,
    value_col="discharge_m3s",
    n=500,
    seed=0,

    # --- peak range control
    enforce_peak_range=True,
    peak_range_m3s=None,              # e.g. (1000.0, 1800.0); if None => uses peak_factor_range
    peak_factor_range=(0.70, 1.20),   # (A,B) = factors * real_peak

    # --- remove valley before rise (normalized-space)
    enforce_no_prepeak_valley=True,
    prepeak_floor_factor=1.0,

    # --- template smoothing
    smooth_real=True,
    real_smooth_window=31,
    real_smooth_poly=3,

    # --- time warp
    warp_a_range=(0.75, 1.35),
    t_split_range=(0.20, 0.40),

    # --- main shape edits
    peak_scale_range=(0.90, 1.15),
    peak_sharp_range=(0.90, 1.25),
    recession_k_range=(0.85, 1.20),

    # --- tail control
    floor_factor_range=(0.90, 1.80),
    floor_strength=4.0,
    end_offset_range=(-0.02, 0.10),
    end_offset_strength=4.0,

    # --- final smoothing
    smooth_syn_window=23,
    smooth_syn_poly=3,

    # --- double peak controls
    p_double=0.7,
    dt2_range=(0.035, 0.10),
    dip_dt_range=(0.015, 0.045),
    dip_depth_range=(0.10, 0.25),
    peak2_amp_range=(0.15, 0.35),
    width_dip_range=(0.010, 0.030),
    width_peak2_range=(0.015, 0.045),

    # --- NEW: force same starting baseflow behavior as the REAL hydrograph
    enforce_real_baseflow_prefix=True,
    baseflow_threshold=0.02,   # prefix ends when Q > base + threshold*(peak-base)
    baseflow_blend_len=12,     # smooth blend from prefix into synthetic
):
    """
    Returns
    -------
    Q_syn_df : DataFrame [nt, n]
        Synthetic discharge in m^3/s with SAME DatetimeIndex as Q_df.
        If enforce_real_baseflow_prefix=True:
          - all synthetics share the same baseflow prefix as the real series (no jump)
    Q_norm_df : DataFrame [nt, n]
        Normalized synthetics in [0,1]
    meta : dict
    """
    rng = np.random.default_rng(seed)

    # ---- coerce input to Series with datetime index
    if isinstance(Q_df, pd.Series):
        q_series = Q_df.copy()
    elif isinstance(Q_df, pd.DataFrame):
        if value_col not in Q_df.columns:
            raise ValueError(f"value_col='{value_col}' not in Q_df.columns={list(Q_df.columns)}")
        q_series = Q_df[value_col].copy()
    else:
        raise TypeError("Q_df must be a pandas DataFrame or Series with a DatetimeIndex.")

    if not isinstance(q_series.index, pd.DatetimeIndex):
        raise TypeError("Q_df must have a DatetimeIndex.")

    Q = q_series.to_numpy(dtype=float)

    # fill NaNs/inf
    if not np.all(np.isfinite(Q)):
        idx0 = np.arange(len(Q))
        good = np.isfinite(Q)
        Q = np.interp(idx0, idx0[good], Q[good])

    # ---- smooth real (template only)
    Q_sm = Q.copy()
    if smooth_real and len(Q_sm) >= 7:
        w = min(real_smooth_window, len(Q_sm) - (1 - len(Q_sm) % 2))
        if w < 7:
            w = 7 if len(Q_sm) >= 7 else (len(Q_sm) // 2) * 2 + 1
        if w % 2 == 0:
            w += 1
        Q_sm = savgol_filter(Q_sm, w, min(real_smooth_poly, w - 1))

    nt = len(Q_sm)
    t = np.linspace(0.0, 1.0, nt)

    Qmin = float(np.min(Q_sm))
    Qmax = float(np.max(Q_sm))
    q0 = (Q_sm - Qmin) / (Qmax - Qmin + 1e-12)
    q_start = float(q0[0])

    base = float(Q[0])
    real_peak = float(np.max(Q))

    # ---- decide A,B
    if peak_range_m3s is None:
        A = float(peak_factor_range[0]) * real_peak
        B = float(peak_factor_range[1]) * real_peak
    else:
        A, B = float(peak_range_m3s[0]), float(peak_range_m3s[1])

    peak_range_m3s_used = (A, B)

    # ---- find REAL baseflow prefix end index (j_base_end)
    # prefix = time steps where Q stays near base before the event begins
    amp_real = max(real_peak - base, 1e-12)
    thr_real = base + baseflow_threshold * amp_real
    above_real = np.where(Q > thr_real)[0]
    j_base_end = int(above_real[0]) if len(above_real) > 0 else min(nt, 10)

    # safety: keep at least 2 points and not too close to end
    j_base_end = int(np.clip(j_base_end, 2, max(2, nt - 3)))

    def time_warp(t, a, t_split):
        s_split = np.clip(a * t_split, 1e-6, 1 - 1e-6)
        slope2 = (1.0 - s_split) / (1.0 - t_split)
        s = np.empty_like(t)
        m1 = t <= t_split
        s[m1] = a * t[m1]
        s[~m1] = s_split + slope2 * (t[~m1] - t_split)
        return np.clip(s, 0.0, 1.0)

    def gauss(t, c, w, a):
        return a * np.exp(-0.5 * ((t - c) / (w + 1e-12))**2)

    def smooth_syn(y):
        if nt < 7:
            return y
        w = min(smooth_syn_window, nt - (1 - nt % 2))
        if w < 7:
            w = 7 if nt >= 7 else (nt // 2) * 2 + 1
        if w % 2 == 0:
            w += 1
        return savgol_filter(y, w, min(smooth_syn_poly, w - 1))

    # -------------------------------------------------
    # 1) generate normalized synthetics
    # -------------------------------------------------
    syn_norm = np.zeros((n, nt), dtype=float)

    for i in range(n):
        a = rng.uniform(*warp_a_range)
        t_split = rng.uniform(*t_split_range)
        s = time_warp(t, a, t_split)
        q = np.interp(s, t, q0)

        q = np.clip(rng.uniform(*peak_scale_range) * q, 0.0, 2.0)
        q = np.clip(q, 0.0, 1.0) ** (1.0 / rng.uniform(*peak_sharp_range))
        q = np.clip(q, 0.0, 1.0)

        # optional double peak
        # optional double peak
        j1 = int(np.argmax(q))
        t1 = t[j1]
        if rng.random() < p_double:
            t_dip = np.clip(t1 + rng.uniform(*dip_dt_range), 0.05, 0.95)
            t2 = np.clip(t1 + rng.uniform(*dt2_range), 0.05, 0.97)
            if t2 <= t_dip + 0.01:
                t2 = np.clip(t_dip + 0.02, 0.05, 0.97)

            q -= gauss(t, t_dip, rng.uniform(*width_dip_range), rng.uniform(*dip_depth_range))
            q += gauss(t, t2,    rng.uniform(*width_peak2_range), rng.uniform(*peak2_amp_range))
            q = np.clip(q, 0.0, 1.0)

        # soft peak capping (apply always)
        q = np.minimum(q, 0.98 + 0.02 * np.tanh((q - 0.98) / 0.02))
        q = np.clip(q, 0.0, 1.0)

            
        # prevent pre-peak valley (normalized)
        j_peak = int(np.argmax(q))
        q = warp_rising_leg(q, j_peak, rng, nt)

        if enforce_no_prepeak_valley and j_peak >= 2:
            floor0 = prepeak_floor_factor * q_start
            pre = q[:j_peak+1].copy()
            pre = np.maximum(pre, floor0)
            pre = np.maximum.accumulate(pre)
            q[:j_peak+1] = pre

        # recession control
        k_rec = rng.uniform(*recession_k_range)
        if j_peak < nt - 2:
            tt = np.linspace(0.0, 1.0, nt - j_peak)
            q[j_peak:] *= (1.0 - tt) ** k_rec

        q[0] = q_start

        # tail floor
        q_floor = rng.uniform(*floor_factor_range) * q_start
        w_floor = (1.0 - q) ** floor_strength
        w_floor[:j_peak] = 0.0
        q += w_floor * np.maximum(0.0, q_floor - q)

        # diverse ends
        w_end = (1.0 - q) ** end_offset_strength
        w_end[:j_peak] = 0.0
        q += w_end * rng.uniform(*end_offset_range)

        q = smooth_syn(np.clip(q, 0.0, 1.0))
        q[0] = q_start
        syn_norm[i] = np.clip(q, 0.0, 1.0)

    # -------------------------------------------------
    # 2) normalized -> discharge
    # -------------------------------------------------
    syn_Q = Qmin + syn_norm * (Qmax - Qmin)
    syn_Q[:, 0] = base

    # -------------------------------------------------
    # 3) peak range enforcement (do it BEFORE baseflow-prefix copy)
    # -------------------------------------------------
    if enforce_peak_range:
        if B <= A:
            raise ValueError(f"Invalid peak range: {peak_range_m3s_used}")
        if B <= base:
            raise ValueError(f"Peak upper bound B={B:.3f} must be > baseflow Q[0]={base:.3f}")

        peaks_now = syn_Q.max(axis=1)
        target_peaks = rng.uniform(A, B, size=n)

        scale = (target_peaks - base) / (peaks_now - base + 1e-12)
        scale = np.clip(scale, 0.1, 10.0)

        syn_Q = base + (syn_Q - base) * scale[:, None]
        syn_Q[:, 0] = base

    # -------------------------------------------------
    # 4) NEW: force ALL synthetics to share REAL baseflow prefix + smooth blend
    # -------------------------------------------------
    if enforce_real_baseflow_prefix and j_base_end >= 2:
        for i in range(n):
            y = syn_Q[i].copy()

            # overwrite prefix with REAL discharge values (same baseflow behavior)
            y[:j_base_end] = Q[:j_base_end]

            # blend next few points so there is no jump at j_base_end
            L = min(baseflow_blend_len, nt - j_base_end)
            if L >= 2:
                w = np.linspace(0.0, 1.0, L)
                y[j_base_end:j_base_end+L] = (1 - w) * Q[j_base_end-1] + w * y[j_base_end:j_base_end+L]

            y[0] = base
            syn_Q[i] = y

        # update normalized to be consistent
        syn_norm = (syn_Q - Qmin) / (Qmax - Qmin + 1e-12)
        syn_norm = np.clip(syn_norm, 0.0, 1.0)

    # -------------------------------------------------
    # 5) pack as DataFrames with SAME datetime index
    # -------------------------------------------------
    idx = q_series.index
    cols = [f"syn_{i:03d}" for i in range(n)]

    Q_syn_df = pd.DataFrame(syn_Q.T, index=idx, columns=cols)
    Q_norm_df = pd.DataFrame(syn_norm.T, index=idx, columns=cols)

    meta = dict(
        Qmin=Qmin, Qmax=Qmax, base=base,
        peak_range_m3s=peak_range_m3s_used,
        seed=seed, n=n, value_col=value_col,
        j_base_end=j_base_end,
        baseflow_threshold=baseflow_threshold,
        baseflow_blend_len=baseflow_blend_len,
        enforce_real_baseflow_prefix=enforce_real_baseflow_prefix,
    )

    return Q_syn_df, Q_norm_df, meta


In [4]:
import numpy as np
import pandas as pd
from scipy.signal import savgol_filter


def generate_synthetic_hydrographs_from_real(
    Q_df,
    value_col="discharge_m3s",
    n=500,
    seed=0,

    # --- peak range control
    enforce_peak_range=True,
    peak_range_m3s=None,              # e.g. (1000.0, 1800.0); if None => uses peak_factor_range
    peak_factor_range=(0.70, 1.20),   # (A,B) = factors * real_peak

    # --- remove valley before rise (normalized-space)
    enforce_no_prepeak_valley=True,
    prepeak_floor_factor=1.0,

    # --- template smoothing
    smooth_real=True,
    real_smooth_window=31,
    real_smooth_poly=3,

    # --- time warp
    warp_a_range=(0.75, 1.35),
    t_split_range=(0.20, 0.40),

    # --- main shape edits
    peak_scale_range=(0.90, 1.15),
    peak_sharp_range=(0.90, 1.25),
    recession_k_range=(0.85, 1.20),

    # --- tail control
    floor_factor_range=(0.90, 1.80),
    floor_strength=4.0,
    end_offset_range=(-0.02, 0.10),
    end_offset_strength=4.0,

    # --- final smoothing
    smooth_syn_window=23,
    smooth_syn_poly=3,

    # --- double peak controls
    p_double=0.7,
    dt2_range=(0.035, 0.10),
    dip_dt_range=(0.015, 0.045),
    dip_depth_range=(0.10, 0.25),
    peak2_amp_range=(0.15, 0.35),
    width_dip_range=(0.010, 0.030),
    width_peak2_range=(0.015, 0.045),

    # --- force same starting baseflow behavior as the REAL hydrograph
    enforce_real_baseflow_prefix=True,
    baseflow_threshold=0.02,   # prefix ends when Q > base + threshold*(peak-base)
    baseflow_blend_len=12,     # smooth blend from prefix into synthetic
):
    """
    Returns
    -------
    Q_syn_df : DataFrame [nt, n]
        Synthetic discharge in m^3/s with SAME DatetimeIndex as Q_df.
        If enforce_real_baseflow_prefix=True:
          - all synthetics share the same baseflow prefix as the real series (no jump)
    Q_norm_df : DataFrame [nt, n]
        Normalized synthetics in [0,1]
    meta : dict
    """
    rng = np.random.default_rng(seed)

    # ---- coerce input to Series with datetime index
    if isinstance(Q_df, pd.Series):
        q_series = Q_df.copy()
    elif isinstance(Q_df, pd.DataFrame):
        if value_col not in Q_df.columns:
            raise ValueError(f"value_col='{value_col}' not in Q_df.columns={list(Q_df.columns)}")
        q_series = Q_df[value_col].copy()
    else:
        raise TypeError("Q_df must be a pandas DataFrame or Series with a DatetimeIndex.")

    if not isinstance(q_series.index, pd.DatetimeIndex):
        raise TypeError("Q_df must have a DatetimeIndex.")

    Q = q_series.to_numpy(dtype=float)

    # fill NaNs/inf
    if not np.all(np.isfinite(Q)):
        idx0 = np.arange(len(Q))
        good = np.isfinite(Q)
        Q = np.interp(idx0, idx0[good], Q[good])

    # ---- smooth real (template only)
    Q_sm = Q.copy()
    if smooth_real and len(Q_sm) >= 7:
        w = min(real_smooth_window, len(Q_sm) - (1 - len(Q_sm) % 2))
        if w < 7:
            w = 7 if len(Q_sm) >= 7 else (len(Q_sm) // 2) * 2 + 1
        if w % 2 == 0:
            w += 1
        Q_sm = savgol_filter(Q_sm, w, min(real_smooth_poly, w - 1))

    nt = len(Q_sm)
    t = np.linspace(0.0, 1.0, nt)

    Qmin = float(np.min(Q_sm))
    Qmax = float(np.max(Q_sm))
    q0 = (Q_sm - Qmin) / (Qmax - Qmin + 1e-12)
    q_start = float(q0[0])

    base = float(Q[0])
    real_peak = float(np.max(Q))

    # ---- decide A,B
    if peak_range_m3s is None:
        A = float(peak_factor_range[0]) * real_peak
        B = float(peak_factor_range[1]) * real_peak
    else:
        A, B = float(peak_range_m3s[0]), float(peak_range_m3s[1])

    peak_range_m3s_used = (A, B)

    # ---- find REAL baseflow prefix end index (j_base_end)
    amp_real = max(real_peak - base, 1e-12)
    thr_real = base + baseflow_threshold * amp_real
    above_real = np.where(Q > thr_real)[0]
    j_base_end = int(above_real[0]) if len(above_real) > 0 else min(nt, 10)
    j_base_end = int(np.clip(j_base_end, 2, max(2, nt - 3)))

    def time_warp(t, a, t_split):
        s_split = np.clip(a * t_split, 1e-6, 1 - 1e-6)
        slope2 = (1.0 - s_split) / (1.0 - t_split)
        s = np.empty_like(t)
        m1 = t <= t_split
        s[m1] = a * t[m1]
        s[~m1] = s_split + slope2 * (t[~m1] - t_split)
        return np.clip(s, 0.0, 1.0)

    def gauss(t, c, w, a):
        return a * np.exp(-0.5 * ((t - c) / (w + 1e-12))**2)

    def smooth_syn(y):
        if nt < 7:
            return y
        w = min(smooth_syn_window, nt - (1 - nt % 2))
        if w < 7:
            w = 7 if nt >= 7 else (nt // 2) * 2 + 1
        if w % 2 == 0:
            w += 1
        return savgol_filter(y, w, min(smooth_syn_poly, w - 1))

    # >>> helpers for time-shift + rising-leg shape diversity
    def shift_series(q, dj, fill_value):
        """Shift by dj time steps; keep same length; fill introduced values."""
        if dj == 0:
            return q
        qq = np.roll(q, dj)
        if dj > 0:
            qq[:dj] = fill_value
        else:
            qq[dj:] = fill_value
        return qq

    def warp_rising_leg(q, j_peak, rng_obj, nt):
        """
        Make rising limb non-linear (avoid straight-line ramps).
        Mix: power warp / logistic warp + tiny roughness, then re-anchor endpoints.
        """
        if j_peak < 8:
            return q

        win = int(np.clip(0.25 * nt, 8, j_peak + 1))
        j0 = max(0, j_peak - win)

        seg = q[j0:j_peak + 1].copy()
        lo = float(seg[0])
        hi = float(seg[-1])
        if hi - lo < 1e-12:
            return q

        u = (seg - lo) / (hi - lo)
        u = np.clip(u, 0.0, 1.0)

        # --- main nonlinearity
        if rng_obj.random() < 0.55:
            pwr = float(rng_obj.uniform(0.15, 3.5))  # wider -> more diverse
            u2 = u ** pwr
        else:
            k = float(rng_obj.uniform(2.5, 16.0))
            m = float(rng_obj.uniform(0.15, 0.85))
            u2 = 1.0 / (1.0 + np.exp(-k * (u - m)))
            u2 = (u2 - u2.min()) / (u2.max() - u2.min() + 1e-12)

        # --- add tiny "wiggle" so it isn't a perfect line (kept small + smooth)
        if len(u2) >= 9:
            amp = float(rng_obj.uniform(0.003, 0.018))
            w1 = float(rng_obj.uniform(1.0, 3.0))
            w2 = float(rng_obj.uniform(2.0, 6.0))
            phi1 = float(rng_obj.uniform(0.0, 2*np.pi))
            phi2 = float(rng_obj.uniform(0.0, 2*np.pi))
            tt = np.linspace(0.0, 1.0, len(u2))
            wig = amp * (np.sin(2*np.pi*w1*tt + phi1) + 0.5*np.sin(2*np.pi*w2*tt + phi2))
            u2 = np.clip(u2 + wig, 0.0, 1.0)

            # smooth wiggle a bit (short SG)
            w = 7 if len(u2) >= 7 else (len(u2)//2)*2+1
            if w >= 5 and w % 2 == 1:
                u2 = savgol_filter(u2, w, 2)

        # re-anchor endpoints exactly
        u2[0] = 0.0
        u2[-1] = 1.0

        q[j0:j_peak + 1] = lo + (hi - lo) * u2
        return q

    # -------------------------------------------------
    # 1) generate normalized synthetics
    # -------------------------------------------------
    syn_norm = np.zeros((n, nt), dtype=float)

    for i in range(n):
        a = rng.uniform(*warp_a_range)
        t_split = rng.uniform(*t_split_range)
        s = time_warp(t, a, t_split)
        q = np.interp(s, t, q0)

        q = np.clip(rng.uniform(*peak_scale_range) * q, 0.0, 2.0)
        q = np.clip(q, 0.0, 1.0) ** (1.0 / rng.uniform(*peak_sharp_range))
        q = np.clip(q, 0.0, 1.0)

        # optional double peak
        j1 = int(np.argmax(q))
        t1 = t[j1]
        if rng.random() < p_double:
            t_dip = np.clip(t1 + rng.uniform(*dip_dt_range), 0.05, 0.95)
            t2 = np.clip(t1 + rng.uniform(*dt2_range), 0.05, 0.97)
            if t2 <= t_dip + 0.01:
                t2 = np.clip(t_dip + 0.02, 0.05, 0.97)

            q -= gauss(t, t_dip, rng.uniform(*width_dip_range), rng.uniform(*dip_depth_range))
            q += gauss(t, t2,    rng.uniform(*width_peak2_range), rng.uniform(*peak2_amp_range))
            q = np.clip(q, 0.0, 1.0)

        # soft peak capping (apply always)
        q = np.minimum(q, 0.98 + 0.02 * np.tanh((q - 0.98) / 0.02))
        q = np.clip(q, 0.0, 1.0)

        # >>> small time-shift so hydrographs are not aligned
        dj = int(rng.integers(-15, 15))
        q = shift_series(q, dj, q_start)

        # >>> different rising-leg slope/shape (the main fix for "perfect lines")
        j_peak = int(np.argmax(q))
        q = warp_rising_leg(q, j_peak, rng, nt)

        # prevent pre-peak valley (normalized)
        if enforce_no_prepeak_valley and j_peak >= 2:
            floor0 = prepeak_floor_factor * q_start
            pre = q[:j_peak+1].copy()
            pre = np.maximum(pre, floor0)
            pre = np.maximum.accumulate(pre)
            q[:j_peak+1] = pre

        # recession control (uses the same j_peak)
        k_rec = rng.uniform(*recession_k_range)
        if j_peak < nt - 2:
            tt = np.linspace(0.0, 1.0, nt - j_peak)
            q[j_peak:] *= (1.0 - tt) ** k_rec

        q[0] = q_start

        # tail floor
        q_floor = rng.uniform(*floor_factor_range) * q_start
        w_floor = (1.0 - q) ** floor_strength
        w_floor[:j_peak] = 0.0
        q += w_floor * np.maximum(0.0, q_floor - q)

        # diverse ends
        w_end = (1.0 - q) ** end_offset_strength
        w_end[:j_peak] = 0.0
        q += w_end * rng.uniform(*end_offset_range)

        q = smooth_syn(np.clip(q, 0.0, 1.0))
        q[0] = q_start
        syn_norm[i] = np.clip(q, 0.0, 1.0)

    # -------------------------------------------------
    # 2) normalized -> discharge
    # -------------------------------------------------
    syn_Q = Qmin + syn_norm * (Qmax - Qmin)
    syn_Q[:, 0] = base

    # -------------------------------------------------
    # 3) peak range enforcement (do it BEFORE baseflow-prefix copy)
    # -------------------------------------------------
    if enforce_peak_range:
        if B <= A:
            raise ValueError(f"Invalid peak range: {peak_range_m3s_used}")
        if B <= base:
            raise ValueError(f"Peak upper bound B={B:.3f} must be > baseflow Q[0]={base:.3f}")

        peaks_now = syn_Q.max(axis=1)
        target_peaks = rng.uniform(A, B, size=n)

        scale = (target_peaks - base) / (peaks_now - base + 1e-12)
        scale = np.clip(scale, 0.1, 10.0)

        syn_Q = base + (syn_Q - base) * scale[:, None]
        syn_Q[:, 0] = base

    # -------------------------------------------------
    # 4) force ALL synthetics to share REAL baseflow prefix + smooth blend
    # -------------------------------------------------
    if enforce_real_baseflow_prefix and j_base_end >= 2:
        for i in range(n):
            y = syn_Q[i].copy()

            # overwrite prefix with REAL discharge values (same baseflow behavior)
            y[:j_base_end] = Q[:j_base_end]

            # blend next few points so there is no jump at j_base_end
            L = min(baseflow_blend_len, nt - j_base_end)
            if L >= 2:
                w = np.linspace(0.0, 1.0, L)
                y[j_base_end:j_base_end+L] = (1 - w) * Q[j_base_end-1] + w * y[j_base_end:j_base_end+L]

            y[0] = base
            syn_Q[i] = y

        # update normalized to be consistent
        syn_norm = (syn_Q - Qmin) / (Qmax - Qmin + 1e-12)
        syn_norm = np.clip(syn_norm, 0.0, 1.0)

    # -------------------------------------------------
    # 5) pack as DataFrames with SAME datetime index
    # -------------------------------------------------
    idx = q_series.index
    cols = [f"syn_{i:03d}" for i in range(n)]

    Q_syn_df = pd.DataFrame(syn_Q.T, index=idx, columns=cols)
    Q_norm_df = pd.DataFrame(syn_norm.T, index=idx, columns=cols)

    meta = dict(
        Qmin=Qmin, Qmax=Qmax, base=base,
        peak_range_m3s=peak_range_m3s_used,
        seed=seed, n=n, value_col=value_col,
        j_base_end=j_base_end,
        baseflow_threshold=baseflow_threshold,
        baseflow_blend_len=baseflow_blend_len,
        enforce_real_baseflow_prefix=enforce_real_baseflow_prefix,
    )

    return Q_syn_df, Q_norm_df, meta


In [5]:
import os
import numpy as np
from netCDF4 import Dataset


# -----------------------------
# Your ANUGA .tms writer (unchanged)
# -----------------------------
def create_tms_manually(file_path, times, discharges):
    """Creates a NetCDF .tms file with cleaned metadata."""
    os.makedirs(os.path.dirname(file_path), exist_ok=True)

    with Dataset(file_path, 'w', format='NETCDF4') as rootgrp:
        rootgrp.createDimension('time', len(times))

        t_var = rootgrp.createVariable('time', 'f8', ('time',))
        q_var = rootgrp.createVariable('discharge', 'f8', ('time',))

        t_var[:] = times
        q_var[:] = discharges

        # Standard ANUGA NetCDF attributes
        rootgrp.description = "Manual conversion for ANUGA"
        rootgrp.starttime = 0.0
        rootgrp.xllcorner = 0.0
        rootgrp.yllcorner = 0.0
        rootgrp.zone = -1
        rootgrp.projection = "UTM"
        rootgrp.datum = "WGS84"


# -----------------------------
# Split 1000 scenarios into N groups (scenario = 8 HGs)
# -----------------------------
def split_scenarios(Q_syn_df_all, N):
    n_stations = len(Q_syn_df_all)
    n_scenarios = Q_syn_df_all[0].shape[1]

    # sanity checks
    for k in range(1, n_stations):
        if Q_syn_df_all[k].shape[1] != n_scenarios:
            raise ValueError("All 8 DataFrames must have the same number of scenario columns.")
        if not Q_syn_df_all[k].index.equals(Q_syn_df_all[0].index):
            raise ValueError("All 8 DataFrames must share the same DatetimeIndex.")

    if n_scenarios % N != 0:
        raise ValueError(f"Number of scenarios ({n_scenarios}) must be divisible by N ({N}).")

    group_size = n_scenarios // N
    scenario_cols = list(Q_syn_df_all[0].columns)

    groups = []
    for g in range(N):
        start = g * group_size
        end = (g + 1) * group_size

        group = []
        for col in scenario_cols[start:end]:
            # one scenario = list of 8 Series (each indexed by datetime)
            scenario = [Q_syn_df_all[s][col].copy() for s in range(n_stations)]
            group.append(scenario)

        groups.append(group)

    return groups


# -----------------------------
# Save groups as .tms files named by station IDs
# -----------------------------
# def save_groups_as_tms(groups, out_dir, station_ids):
#     """
#     groups[g][s][k] is a pandas Series:
#       g = group index
#       s = scenario index within group  (NOW starts from 0 for each group)
#       k = station index (0..7)

#     Writes:
#       out_dir/group_000/scenario_000000/<station_id>.tms
#       out_dir/group_001/scenario_000000/<station_id>.tms
#       ...
#     """
#     if len(station_ids) != 1:
#         raise ValueError("station_ids must have length 8.")

#     os.makedirs(out_dir, exist_ok=True)

#     for g, group in enumerate(groups):
#         group_dir = os.path.join(out_dir, f"group_{g:03d}")
#         os.makedirs(group_dir, exist_ok=True)

#         for s, scenario in enumerate(group):
#             # --- CHANGED: scenario id resets within each group
#             scen_dir = os.path.join(group_dir, f"scenario_{s:06d}")
#             os.makedirs(scen_dir, exist_ok=True)

#             for k, hg_series in enumerate(scenario):
#                 idx = hg_series.index
#                 t0 = idx[0]
#                 times_sec = (idx - t0).total_seconds().astype(float)
#                 discharges = hg_series.to_numpy(dtype=float)

#                 station_id = str(station_ids[k])
#                 file_path = os.path.join(scen_dir, f"tms_files/{station_id}.tms")

#                 create_tms_manually(file_path, times_sec, discharges)

import os

def save_groups_as_tms(groups, out_dir, station_ids):
    """
    groups[g][s][k] is a pandas Series:
      g = group index
      s = scenario index within group
      k = station index (0..7)

    Writes:
      out_dir/group_000/scenario_000000/<station_id>.tms
    """
    # Note: Logic remains for station_ids provided by user
    if len(station_ids) != 1:
        raise ValueError("station_ids must have length 1.")

    os.makedirs(out_dir, exist_ok=True)

    for g, group in enumerate(groups):
        group_dir = os.path.join(out_dir, f"group_{g:03d}")
        os.makedirs(group_dir, exist_ok=True)

        for s, scenario in enumerate(group):
            scen_dir = os.path.join(group_dir, f"scenario_{s:06d}")
            # Ensure the tms_files subdirectory exists as expected by your path logic
            tms_dir = os.path.join(scen_dir, "tms_files")
            os.makedirs(tms_dir, exist_ok=True)

            for k, hg_series in enumerate(scenario):
                # --- UPDATED: index is already in seconds ---
                idx = hg_series.index.to_numpy()
                t0 = idx[0]
                # Subtract t0 to ensure the TMS starts at time 0.0
                times_sec = (idx - t0).astype(float)
                discharges = hg_series.to_numpy(dtype=float)

                station_id = str(station_ids[k])
                file_path = os.path.join(tms_dir, f"{station_id}.tms")

                create_tms_manually(file_path, times_sec, discharges)

In [6]:
# gauges, labels, usgs_ids, dfs, Qs, nts, t_days_list = load_gauge_and_discharge(
#     gauge_shp_dir="/storage/group/cxs1024/default/mehdi/Hurricane_MatthewData/UpstreamLocations",
#     gauge_shp_name="UpstreamBoundry_locations.shp",
#     discharge_folder="/storage/group/cxs1024/default/mehdi/Hurricane_MatthewData/discharge_1main&7tributaries",
#     gauge_index=None,  # This triggers "all gauges" mode
# )


In [7]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

SEED  = 44
station_ids = [1]

N = 500  # Number of random scenarios
N_GROUPS = 13  # Example



# 1. Configuration
# Assigning RANDOM SEED for reproducibility
np.random.seed(SEED)  

# 2. Define Parameter Ranges (Scenario S1)
# Qp range: [6660.5, 19981.5] m^3/s
# tp range: [2.0, 6.0] hours
# T range:  [15.0, 20.0] hours

# Range_1 (Group_1 and Group_UQ)
# Qp_min, Qp_max = 6500, 20000
# tp_min, tp_max = 2.0, 6.0    
# T_min, T_max   = 15.0, 20.0  

# # Range_2 (Group_2 Group_UQ2)
Qp_min, Qp_max = 10_000, 15_000
tp_min, tp_max = 4.0, 6.0    
T_min, T_max   = 15.0, 20.0  


# 3. Generate N random combinations
random_params = pd.DataFrame({
    'Qp': np.random.uniform(Qp_min, Qp_max, N),
    'tp': np.random.uniform(tp_min, tp_max, N),
    'T':  np.random.uniform(T_min, T_max, N)
})

# 4. Define Time Array (0 to 24 hours in seconds)
time_s = np.arange(0, 24 * 3600 + 1, 1)

# 5. Define the triangular hydrograph logic
def calculate_q(t, Qp, tp_h, T_h):
    tp_s = tp_h * 3600 # hours to seconds
    T_s = T_h * 3600   # hours to seconds
    
    if t <= tp_s:
        # Linear ramp up to peak
        return (Qp / tp_s) * t
    elif t <= T_s:
        # Linear recession to zero
        return Qp - ((Qp / (T_s - tp_s)) * (t - tp_s))
    else:
        # Flow is finished
        return 0.0

# 6. Build the DataFrame
data = {'time_s': time_s}

for i in range(N):
    col_name = f'syn_{i:03d}'
    p = random_params.iloc[i]
    # Calculate Q values based on synthetic breach hydrograph logic
    data[col_name] = [calculate_q(t, p['Qp'], p['tp'], p['T']) for t in time_s]

# Create the final DataFrame and set time_s as index to avoid an extra index column
df = [pd.DataFrame(data).set_index('time_s')]

In [8]:
MAIN_PATH = '/storage/group/cxs1024/default/mehdi/dam_break/'

groups = split_scenarios(df, 20)

save_groups_as_tms(
    groups,
    out_dir=MAIN_PATH + "scenario_groups_UQ2",
    station_ids=station_ids
)


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

def plot_all_samples_per_gauge(Q_syn_df_all, station_ids, max_plot=None):
    """
    Plots all synthetic hydrographs for each gauge separately and prints Q stats.
    """
    
    # Track global min/max across all gauges and samples
    global_min = float('inf')
    global_max = float('-inf')

    for i, Q_syn_df in enumerate(Q_syn_df_all):
        sid = station_ids[i]
        
        # Update global statistics
        current_min = Q_syn_df.min().min()
        current_max = Q_syn_df.max().max()
        global_min = min(global_min, current_min)
        global_max = max(global_max, current_max)

        # Plotting logic
        plt.figure(figsize=(11, 4))
        cols = Q_syn_df.columns
        if max_plot is not None:
            cols = cols[:max_plot]

        for col in cols:
            plt.plot(
                Q_syn_df.index,
                Q_syn_df[col],
                color="C0",
                alpha=0.9  # Reduced alpha for better visibility when plotting hundreds of lines
            )

        plt.title(f"All synthetic hydrographs – Gauge {sid}")
        plt.xlabel("Time")
        plt.ylabel("Discharge (m$^3$/s)")
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
        
        print(f"Gauge {sid}: Min Q = {current_min:.2f}, Max Q = {current_max:.2f}")

    print("\n" + "="*30)
    print(f"GLOBAL STATS FOR ALL SAMPLES")
    print(f"Absolute Min Q: {global_min:.2f} m^3/s")
    print(f"Absolute Max Q: {global_max:.2f} m^3/s")
    print("="*30)

# Execution
plot_all_samples_per_gauge(
    df,
    station_ids,
    max_plot=500 
)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from netCDF4 import Dataset


def load_one_scenario_tms(base_dir, group_id, scenario_idx, station_ids):
    """
    Loads ONE scenario by index inside a group.

    Directory structure (UPDATED):
      base_dir/
        group_XXX/
          scenario_000000/
          scenario_000001/
          ...

    Parameters
    ----------
    base_dir : str
    group_id : int
        Group index (e.g. 0 -> group_000)
    scenario_idx : int
        Scenario index INSIDE the group (starts from 0)
    station_ids : list[int]

    Returns
    -------
    data : dict
        {station_id: (time_sec, discharge)}
    scen_name : str
        Folder name (e.g. 'scenario_000008')
    """

    group_dir = os.path.join(base_dir, f"group_{group_id:03d}")
    if not os.path.isdir(group_dir):
        raise FileNotFoundError(f"Group folder not found: {group_dir}")

    # discover scenario folders inside the group
    scenarios = sorted(
        d for d in os.listdir(group_dir)
        if d.startswith("scenario_")
    )

    if len(scenarios) == 0:
        raise RuntimeError(f"No scenarios found in {group_dir}")

    if scenario_idx < 0 or scenario_idx >= len(scenarios):
        raise IndexError(
            f"scenario_idx={scenario_idx} out of range (0–{len(scenarios)-1})"
        )

    scen_name = scenarios[scenario_idx]
    scen_dir = os.path.join(group_dir, scen_name)

    data = {}
    for sid in station_ids:
        fpath = os.path.join(scen_dir, f"tms_files/{sid}.tms")
        if not os.path.exists(fpath):
            raise FileNotFoundError(f"Missing .tms file: {fpath}")

        with Dataset(fpath, "r") as ds:
            t = np.asarray(ds.variables["time"][:], dtype=float)
            q = np.asarray(ds.variables["discharge"][:], dtype=float)

        data[str(sid)] = (t, q)

    return data, scen_name
def plot_scenario_hgs(data, title=None, time_unit="hours"):
    """
    data : dict {station_id: (time_sec, discharge)}
    """
    plt.figure(figsize=(11, 5))

    for sid, (t, q) in data.items():
        if time_unit == "hours":
            x = t / 3600.0
            xlabel = "Time (hours)"
        elif time_unit == "days":
            x = t / (24.0 * 3600.0)
            xlabel = "Time (days)"
        else:
            x = t
            xlabel = "Time (seconds)"

        plt.plot(x, q, label=str(sid), alpha=0.9)

    plt.xlabel(xlabel)
    plt.ylabel("Discharge (m$^3$/s)")
    plt.grid(True, alpha=0.3)
    if title is not None:
        plt.title(title)
    plt.legend(ncol=2, fontsize=9)
    plt.tight_layout()
    plt.show()


In [ ]:
station_ids = [1]

base_dir = MAIN_PATH + "scenario_groups_1"
group_id = 2        # group_000
scenario_idx = 5    # scenario_000008 INSIDE group_000

data, scen_name = load_one_scenario_tms(
    base_dir,
    group_id,
    scenario_idx,
    station_ids
)

plot_scenario_hgs(
    data,
    title=f"group_{group_id:03d} / {scen_name}",
    time_unit="hours"
)


In [ ]:
station_ids = [1]

base_dir = MAIN_PATH + "scenario_groups_1"
group_id = 20        # group_000
scenario_idx = 5    # scenario_000008 INSIDE group_000

data, scen_name = load_one_scenario_tms(
    base_dir,
    group_id,
    scenario_idx,
    station_ids
)

plot_scenario_hgs(
    data,
    title=f"group_{group_id:03d} / {scen_name}",
    time_unit="hours"
)
